# Stock Price Trend Prediction: Chapter 2A

## Data Preperation For DL Models — Feature Engineering, Scaling & Sequence Window Creation

<br>

This notebook takes the raw extracted features produced in **Chapter 0** and prepares them
for the CNN + LSTM model. It performs **three major tasks**:

1. **Feature Engineering** — drop redundant/leakage columns, modify weak features, and add new ones in parallel
2. **Scaling** — apply MinMaxScaler fitted _only_ on the training set
3. **Sequence Window Creation** — reshape the flat time-series into 3-D sliding windows `(samples, timesteps, features)`

All computationally heavy steps use `joblib` / `multiprocessing` to saturate your CPU cores.

<br>


### Part A: Import all libraries and load the preprocessed dataset.


In [14]:
# A1 — Imports & CPU Detection
import os
import pickle
import warnings
import multiprocessing

import numpy  as np
import pandas as pd
from joblib             import Parallel, delayed
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings('ignore')

N_CORES = max(1, multiprocessing.cpu_count() - 1)

print(f'✅ Python environment ready')
print(f'   Logical CPU cores detected : {multiprocessing.cpu_count()}')
print(f'   Cores reserved for workers : {N_CORES}  (1 kept free for OS)')


✅ Python environment ready
   Logical CPU cores detected : 16
   Cores reserved for workers : 15  (1 kept free for OS)


In [15]:
# A2 — Load the Feature-Extracted CSV from Chapter 0

df = pd.read_csv('../Chapter 0: Feature Extraction From Dataset/Final_Extracted_Features.csv', index_col=0, parse_dates=True)

print(f'✅ Dataset loaded successfully!')
print(f'\n   Shape     : {df.shape[0]:,} rows  ×  {df.shape[1]} columns')
print(f'   Date range: {df.index.min().date()}  →  {df.index.max().date()}')
print(f'\n   Columns ({len(df.columns)}):')
for i, col in enumerate(df.columns, 1):
    marker = '🎯' if col == 'target' else '  '
    print(f'   {marker}  {i:02d}. {col:<20}  dtype: {str(df[col].dtype):<10}  '
          f'nulls: {df[col].isnull().sum()}')

print(f'\n--- Target Class Balance ---')
vc = df['target'].value_counts(normalize=True)
print(f'   UP   (1) : {vc.get(1, 0):.2%}')
print(f'   DOWN (0) : {vc.get(0, 0):.2%}')

print('\n--- First 3 Rows ---')
df.head(3)


✅ Dataset loaded successfully!

   Shape     : 3,804 rows  ×  19 columns
   Date range: 2010-02-01  →  2025-03-14

   Columns (19):
       01. open                  dtype: float64     nulls: 0
       02. high                  dtype: float64     nulls: 0
       03. low                   dtype: float64     nulls: 0
       04. close                 dtype: float64     nulls: 0
       05. adj_close             dtype: float64     nulls: 0
       06. volume                dtype: int64       nulls: 0
       07. ma5                   dtype: float64     nulls: 0
       08. ma20                  dtype: float64     nulls: 0
       09. ma_cross              dtype: int64       nulls: 0
       10. return                dtype: float64     nulls: 0
       11. volume_change         dtype: float64     nulls: 0
       12. rsi14                 dtype: float64     nulls: 0
       13. bb_mid                dtype: float64     nulls: 0
       14. bb_upper              dtype: float64     nulls: 0
       15. bb_

,open,high,low,close,adj_close,volume,ma5,ma20,ma_cross,return,volume_change,rsi14,bb_mid,bb_upper,bb_lower,bb_width,macd,signal,target
date,,,,,,,,,,,,,,,,,,,
2010-02-01,6.870357,7.000000,6.832143,6.954643,5.860126,749876400,6.018117,6.231361,0,0.013902,-0.398150,38.180145,6.231361,6.624758,5.837963,0.126264,-0.074927,-0.041260,1
2010-02-02,6.996786,7.011429,6.906429,6.995000,5.894133,698342400,5.957449,6.204051,0,0.005803,-0.068723,40.705386,6.204051,6.611935,5.796167,0.131490,-0.080788,-0.049239,1
2010-02-03,6.970357,7.150000,6.943571,7.115357,5.995550,615328000,5.905387,6.181255,0,0.017206,-0.118873,41.111516,6.181255,6.581813,5.780697,0.129604,-0.077561,-0.054946,0


<br><br>

### Part B: Feature Engineering — Modify, Add, and Clean.


In [16]:
# ============================================================
#  B1 — Modify Existing Features
#
#  1. ma_cross  →  ma_cross_strength  (continuous)
#
#  2. bb_upper + bb_lower  →  bb_position (0–1)
#
#  3. macd + signal  →  add macd_hist
#
#  4. ma5 + ma20  →  ma5_dist + ma20_dist  (scale-free)
#     WHY THIS CHANGE:
#     ma5 and ma20 are absolute dollar moving averages.
#     AAPL was ~$7 in 2010 and ~$170 in 2022. After MinMaxScaler
#     fits on 2010–2020, the 2021–2022 val values reach 1.36–1.41
#     — completely outside the [0,1] training range. The model's
#     neurons saturate and it predicts all-DOWN every epoch.
#     Fix: replace with DISTANCE from price to MA, as a fraction
#     of price. This is scale-free — works identically at $7 or $170.
#       ma5_dist  = (adj_close - ma5)  / adj_close
#       ma20_dist = (adj_close - ma20) / adj_close
#     Positive = price above MA (bullish), Negative = below (bearish).
#     ma_cross_strength already captures the MA5 vs MA20 gap,
#     so ma5_dist and ma20_dist add independent price-vs-MA signal.
#
#  5. bb_mid dropped — identical to ma20.
# ============================================================

data = df.copy()

# 1. MA cross strength — scale-free, continuous, gradient-friendly
data['ma_cross_strength'] = (data['ma5'] - data['ma20']) / data['ma20']
data.drop(columns=['ma_cross'], inplace=True)

# 2. BB position — where is price within the band? (0=bottom, 1=top)
data['bb_position'] = (
    (data['adj_close'] - data['bb_lower'])
    / (data['bb_upper'] - data['bb_lower'])
)
data.drop(columns=['bb_upper', 'bb_lower', 'bb_mid'], inplace=True)

# 3. MACD histogram — divergence speed
data['macd_hist'] = data['macd'] - data['signal']

# 4. ── NEW: Replace ma5/ma20 absolute values with scale-free distances ──
#    These replace the raw moving averages which caused val features
#    to reach 1.36–1.41 after MinMaxScaling, breaking the model entirely.
data['ma5_dist']  = (data['adj_close'] - data['ma5'])  / data['adj_close']
data['ma20_dist'] = (data['adj_close'] - data['ma20']) / data['adj_close']
data.drop(columns=['ma5', 'ma20'], inplace=True)

print('✅ Feature modifications applied!')
print(f'\n   ma_cross          →  ma_cross_strength')
print(f'     range : [{data["ma_cross_strength"].min():.5f},  {data["ma_cross_strength"].max():.5f}]')
print(f'\n   bb_upper + bb_lower + bb_mid  →  bb_position')
print(f'     range : [{data["bb_position"].min():.4f},  {data["bb_position"].max():.4f}]')
print(f'\n   macd + signal  →  added macd_hist = macd − signal')
print(f'     range : [{data["macd_hist"].min():.5f},  {data["macd_hist"].max():.5f}]')
print(f'\n   ma5 (absolute $)  →  ma5_dist  = (close - ma5)  / close')
print(f'     range : [{data["ma5_dist"].min():.5f},  {data["ma5_dist"].max():.5f}]')
print(f'\n   ma20 (absolute $) →  ma20_dist = (close - ma20) / close')
print(f'     range : [{data["ma20_dist"].min():.5f},  {data["ma20_dist"].max():.5f}]')
print(f'\nCurrent shape: {data.shape}')

✅ Feature modifications applied!

   ma_cross          →  ma_cross_strength
     range : [-0.11162,  0.10741]

   bb_upper + bb_lower + bb_mid  →  bb_position
     range : [-0.3056,  1.3701]

   macd + signal  →  added macd_hist = macd − signal
     range : [-3.34124,  2.33655]

   ma5 (absolute $)  →  ma5_dist  = (close - ma5)  / close
     range : [-0.09952,  0.08610]

   ma20 (absolute $) →  ma20_dist = (close - ma20) / close
     range : [-0.20197,  0.12854]

Current shape: (3804, 18)


In [17]:
# ============================================================
#  B2 — Parallel Computation of 4 New Features
#
#  ATR-14  (Average True Range) — unchanged
#    True volatility including overnight gaps.
#
#  OBV → obv_change  (Rolling Z-Score, UPDATED)
#    Previous version (pct_change) produced val range [0.37, 0.40]
#    — a spread of only 0.03 after scaling. Nearly zero signal.
#    New version: 20-day rolling z-score of OBV.
#      z = (OBV - rolling_mean_20) / rolling_std_20
#    This is stationary across all time periods — a z-score of
#    +2 means OBV is 2 std above its recent mean regardless of
#    whether AAPL is at $7 or $170. Clipped to [-3, 3] to
#    suppress extreme outlier days. MinMaxScaler will then map
#    this well-behaved range cleanly to [0, 1].
#
#  body_ratio  (Candlestick Body Strength) — unchanged
#    (close − open) / (high − low) → range: −1 to +1
#
#  day_of_week  (Calendar Effect) — unchanged
#    0=Mon … 4=Fri.
# ============================================================

def _compute_atr(df_local, period=14):
    """Average True Range — true volatility including overnight gaps."""
    prev_close = df_local['adj_close'].shift(1)
    tr = pd.concat([
        df_local['high'] - df_local['low'],
        (df_local['high'] - prev_close).abs(),
        (df_local['low']  - prev_close).abs()
    ], axis=1).max(axis=1)
    return tr.rolling(period).mean().rename('atr14')


def _compute_obv_change(df_local, window=20):
    """OBV rolling z-score — stationary, scale-free volume flow signal.

    Raw OBV is cumulative and grows monotonically over 15 years,
    causing extreme compression after MinMaxScaling on train data.
    The 20-day rolling z-score solves this:
      z = (OBV - rolling_mean_20) / rolling_std_20
    A positive z means volume flow is above its recent norm (bullish),
    negative means below (bearish). Scale-free at any price level.
    Clipped to [-3, 3] to suppress extreme outlier days.
    NaN rows (first 20) filled with 0 (neutral signal).
    """
    direction         = np.sign(df_local['adj_close'].diff())
    direction.iloc[0] = 0
    obv      = (direction * df_local['volume']).cumsum()
    obv_mean = obv.rolling(window).mean()
    obv_std  = obv.rolling(window).std().replace(0, np.nan)
    z        = ((obv - obv_mean) / obv_std).fillna(0).clip(-3, 3)
    return z.rename('obv_change')


def _compute_body_ratio(df_local):
    """Candlestick body ratio — intraday momentum direction & strength."""
    body   = df_local['adj_close'] - df_local['open']
    candle = (df_local['high'] - df_local['low']).replace(0, np.nan)
    return (body / candle).rename('body_ratio')


def _compute_dow(df_local):
    """Day of week — 0 (Monday) to 4 (Friday)."""
    return pd.Series(
        df_local.index.dayofweek,
        index=df_local.index,
        name='day_of_week'
    )


FEATURE_FNS = [_compute_atr, _compute_obv_change, _compute_body_ratio, _compute_dow]
n_parallel  = min(N_CORES, len(FEATURE_FNS))

print(f'⏳ Computing {len(FEATURE_FNS)} features in parallel across {n_parallel} cores...')
print(f'   Note: OBV replaced with 20-day rolling z-score (clipped ±3)\n')

results = Parallel(n_jobs=n_parallel)(
    delayed(fn)(data) for fn in FEATURE_FNS
)

for series in results:
    data[series.name] = series.values

print('✅ Parallel feature computation complete!')
print(f'\n   atr14       — range : [{data["atr14"].min():.4f},  {data["atr14"].max():.4f}]')
print(f'   obv_change  — range : [{data["obv_change"].min():.4f},  {data["obv_change"].max():.4f}]')
print(f'   body_ratio  — range : [{data["body_ratio"].min():.4f},  {data["body_ratio"].max():.4f}]')
print(f'   day_of_week — unique : {sorted(data["day_of_week"].unique())}')
print(f'\nCurrent shape: {data.shape}')

⏳ Computing 4 features in parallel across 4 cores...
   Note: OBV replaced with 20-day rolling z-score (clipped ±3)

✅ Parallel feature computation complete!

   atr14       — range : [1.1172,  7.6631]
   obv_change  — range : [-3.0000,  3.0000]
   body_ratio  — range : [-38.6097,  0.9465]
   day_of_week — unique : [np.int32(0), np.int32(1), np.int32(2), np.int32(3), np.int32(4)]

Current shape: (3804, 22)


In [18]:
# ============================================================
#  B3 — Drop Raw Price Columns  (Leakage Prevention)
#
#  open, high, low, close, adj_close are absolute dollar values.
#  AAPL was ~$7 in 2010 and ~$200 in 2025. If these live inside
#  the model's feature matrix, the CNN will learn to recognise
#  *which historical era* the window belongs to, not *what the
#  price pattern means*. This is called price-level leakage.
#
#  We have already extracted all value from them:
#    adj_close  →  ma5, ma20, return, rsi14, bb_*, macd, atr14, obv
#    open       →  body_ratio, ma_cross_strength
#    high/low   →  atr14, body_ratio, bb_position
#
#  'volume' is kept — it is not a price, it is a count. Its
#  scale variation is handled cleanly by the MinMaxScaler.
# ============================================================

LEAKAGE_COLS = ['open', 'high', 'low', 'close', 'adj_close']

print('Dropping raw price columns (leakage prevention):')
for col in LEAKAGE_COLS:
    print(f'   ✗  {col}')

data.drop(columns=LEAKAGE_COLS, inplace=True)

print(f'\n✅ Done.  Shape after drop: {data.shape}')
print(f'\nFinal feature set ({len(data.columns)} columns):')
for i, col in enumerate(data.columns, 1):
    marker = '🎯' if col == 'target' else '  '
    print(f'  {marker}  {i:02d}.  {col}')


Dropping raw price columns (leakage prevention):
   ✗  open
   ✗  high
   ✗  low
   ✗  close
   ✗  adj_close

✅ Done.  Shape after drop: (3804, 17)

Final feature set (17 columns):
      01.  volume
      02.  return
      03.  volume_change
      04.  rsi14
      05.  bb_width
      06.  macd
      07.  signal
  🎯  08.  target
      09.  ma_cross_strength
      10.  bb_position
      11.  macd_hist
      12.  ma5_dist
      13.  ma20_dist
      14.  atr14
      15.  obv_change
      16.  body_ratio
      17.  day_of_week


In [19]:
# ============================================================
#  B4 — Feature Audit: NaN, Inf, Balance, Stats
#
#  ATR-14 introduces 13 NaN rows at the top (rolling window),
#  and body_ratio can produce NaN where high == low (rare).
#  We drop any remaining NaN / Inf rows here before splitting.
#
#  Checks performed:
#    1. NaN count per column — drop affected rows if any
#    2. Infinite values  — replace with NaN then drop
#    3. Target class balance after engineering
#    4. Descriptive stats for a final sanity check
# ============================================================

print('--- 1. NaN Check ---')
nan_counts = data.isnull().sum()
has_nan    = nan_counts[nan_counts > 0]
if len(has_nan) == 0:
    print('   ✅ No NaN values found.')
else:
    print(f'   ⚠️  NaN detected in {len(has_nan)} column(s):')
    print(has_nan.to_string())
    rows_before = len(data)
    data.dropna(inplace=True)
    print(f'   Dropped {rows_before - len(data)} rows → new shape: {data.shape}')

print('\n--- 2. Inf Check ---')
inf_mask = np.isinf(data.select_dtypes(include=np.number)).any()
inf_cols  = inf_mask[inf_mask].index.tolist()
if inf_cols:
    print(f'   ⚠️  Inf values in: {inf_cols}  — replacing with NaN then dropping')
    data.replace([np.inf, -np.inf], np.nan, inplace=True)
    data.dropna(inplace=True)
    print(f'   Shape after inf removal: {data.shape}')
else:
    print('   ✅ No infinite values found.')

print('\n--- 3. Target Class Balance (post-engineering) ---')
vc = data['target'].value_counts(normalize=True)
print(f'   UP   (1) : {vc.get(1, 0):.2%}')
print(f'   DOWN (0) : {vc.get(0, 0):.2%}')
balanced = abs(vc.get(1, 0) - vc.get(0, 0)) < 0.10
print(f'   {"✅ Balanced — no class weighting needed" if balanced else "⚠️  Imbalanced — consider class_weight in model"}')

print('\n--- 4. Descriptive Stats (features only) ---')
feature_cols_tmp = [c for c in data.columns if c != 'target']
print(data[feature_cols_tmp].describe().round(5).to_string())
print(f'\n✅ Audit complete.  Final shape: {data.shape}')
print(f'   Date range: {data.index.min().date()}  →  {data.index.max().date()}')


--- 1. NaN Check ---
   ⚠️  NaN detected in 1 column(s):
atr14    13
   Dropped 13 rows → new shape: (3791, 17)

--- 2. Inf Check ---
   ✅ No infinite values found.

--- 3. Target Class Balance (post-engineering) ---
   UP   (1) : 52.99%
   DOWN (0) : 47.01%
   ✅ Balanced — no class weighting needed

--- 4. Descriptive Stats (features only) ---
             volume      return  volume_change       rsi14    bb_width        macd      signal  ma_cross_strength  bb_position   macd_hist    ma5_dist   ma20_dist       atr14  obv_change  body_ratio  day_of_week
count  3.791000e+03  3791.00000     3791.00000  3791.00000  3791.00000  3791.00000  3791.00000         3791.00000   3791.00000  3791.00000  3791.00000  3791.00000  3791.00000  3791.00000  3791.00000   3791.00000
mean   2.235352e+08     0.00109        0.05159    55.95111     0.11637     0.42464     0.42693            0.00745      0.58640    -0.00228     0.00158     0.00774     3.28883     0.11410    -4.89101      2.02532
std    2.114048e+

<br><br>

### Part C: Temporal Train / Validation / Test Split.


In [20]:
# ============================================================
#  C1 — Temporal Train / Validation / Test Split
#
#  Stock data is a time series — random shuffling MUST NOT be
#  used. Shuffling allows future rows into the training set,
#  which is a form of lookahead bias / data leakage.
#
#  Split boundaries (chosen to cover distinct market regimes):
#
#    Train : 2010-01-01 → 2020-12-31  (~10 years)
#      Covers the post-GFC recovery, bull run, 2018 correction,
#      and the COVID crash + recovery of 2020.
#
#    Val   : 2021-01-01 → 2022-12-31  (~2 years)
#      Post-COVID meme-stock mania, rate hike cycle, 2022 bear.
#      Used for hyperparameter tuning and early stopping only.
#
#    Test  : 2023-01-01 → end of data  (~2+ years)
#      AI boom rally, volatility clusters.
#      Touched ONCE — at final evaluation.
#
#  The scaler will be fit ONLY on the training set rows.
# ============================================================

TRAIN_END = '2020-12-31'
VAL_END   = '2022-12-31'

train_df = data.loc[data.index <= TRAIN_END].copy()
val_df   = data.loc[(data.index > TRAIN_END) & (data.index <= VAL_END)].copy()
test_df  = data.loc[data.index > VAL_END].copy()

# Verify no gaps or overlaps
assert train_df.index.max() < val_df.index.min(),   'Train/Val overlap!'
assert val_df.index.max()   < test_df.index.min(),  'Val/Test overlap!'

total = len(data)
print('✅ Temporal split complete  (no shuffling, no overlap)')
print(f'\n   Split       Date Range                            Rows    Share')
print(f'   {"─"*65}')
print(f'   Train     {train_df.index.min().date()} → {train_df.index.max().date()}'
      f'    {len(train_df):>5,}   {len(train_df)/total:.0%}')
print(f'   Val       {val_df.index.min().date()} → {val_df.index.max().date()}'
      f'    {len(val_df):>5,}   {len(val_df)/total:.0%}')
print(f'   Test      {test_df.index.min().date()} → {test_df.index.max().date()}'
      f'    {len(test_df):>5,}   {len(test_df)/total:.0%}')
print(f'   {"─"*65}')
print(f'   Total                                         {total:>5,}   100%')


✅ Temporal split complete  (no shuffling, no overlap)

   Split       Date Range                            Rows    Share
   ─────────────────────────────────────────────────────────────────
   Train     2010-02-19 → 2020-12-31    2,737   72%
   Val       2021-01-04 → 2022-12-30      503   13%
   Test      2023-01-03 → 2025-03-14      551   15%
   ─────────────────────────────────────────────────────────────────
   Total                                         3,791   100%


In [21]:
# ============================================================
#  C2 — Separate Features from Target Labels
#
#  The target column is separated *before* scaling so the
#  MinMaxScaler never touches the 0/1 labels. Labels remain
#  integer arrays throughout — no floating-point contamination.
# ============================================================

FEATURE_COLS = [c for c in data.columns if c != 'target']
TARGET_COL   = 'target'

X_train_raw = train_df[FEATURE_COLS].values.astype(np.float64)
y_train_raw = train_df[TARGET_COL].values.astype(np.int8)

X_val_raw   = val_df[FEATURE_COLS].values.astype(np.float64)
y_val_raw   = val_df[TARGET_COL].values.astype(np.int8)

X_test_raw  = test_df[FEATURE_COLS].values.astype(np.float64)
y_test_raw  = test_df[TARGET_COL].values.astype(np.int8)

print('✅ Feature / target separation complete!')
print(f'\n   Feature columns ({len(FEATURE_COLS)}):')
for i, col in enumerate(FEATURE_COLS, 1):
    print(f'     {i:02d}.  {col}')
print(f'\n   X_train_raw : {X_train_raw.shape}   y_train_raw : {y_train_raw.shape}')
print(f'   X_val_raw   : {X_val_raw.shape}     y_val_raw   : {y_val_raw.shape}')
print(f'   X_test_raw  : {X_test_raw.shape}    y_test_raw  : {y_test_raw.shape}')


✅ Feature / target separation complete!

   Feature columns (16):
     01.  volume
     02.  return
     03.  volume_change
     04.  rsi14
     05.  bb_width
     06.  macd
     07.  signal
     08.  ma_cross_strength
     09.  bb_position
     10.  macd_hist
     11.  ma5_dist
     12.  ma20_dist
     13.  atr14
     14.  obv_change
     15.  body_ratio
     16.  day_of_week

   X_train_raw : (2737, 16)   y_train_raw : (2737,)
   X_val_raw   : (503, 16)     y_val_raw   : (503,)
   X_test_raw  : (551, 16)    y_test_raw  : (551,)


<br><br>

### Part D: Feature Scaling — MinMaxScaler (fit on train only).


In [22]:
# ============================================================
#  D1 — MinMaxScaler: Fit on Train, Transform All
#
#  GOLDEN RULE: fit the scaler ONLY on training rows.
#
#  If you fit on the full dataset, the scaler learns the min/max
#  of 2025 data while training on 2010 data — the model indirectly
#  'sees' the future distribution. This is data leakage.
#
#  MinMaxScaler maps each feature column to [0, 1]:
#    x_scaled = (x − x_min_train) / (x_max_train − x_min_train)
#
#  Val and test values may fall slightly outside [0, 1] if they
#  exceed the training range — this is correct and expected
#  behaviour, not a bug.
#
#  The fitted scaler is saved so it can be reloaded identically
#  during live inference without re-fitting from scratch.
# ============================================================

scaler = MinMaxScaler(feature_range=(0, 1))

# Fit ONLY on training set
scaler.fit(X_train_raw)

# Apply to all three splits
X_train_scaled = scaler.transform(X_train_raw)
X_val_scaled   = scaler.transform(X_val_raw)
X_test_scaled  = scaler.transform(X_test_raw)

print('✅ Scaling complete!')
print(f'   Scaler fitted on {X_train_raw.shape[0]:,} training rows only.')
print(f'\n   Post-scaling ranges (training set):')
print(f'   {"Feature":<22} {"Min":>8} {"Max":>8}')
print(f'   {"─"*40}')
for i, col in enumerate(FEATURE_COLS):
    col_min = X_train_scaled[:, i].min()
    col_max = X_train_scaled[:, i].max()
    flag = '  ⚠️ ' if col_max > 1.01 or col_min < -0.01 else ''
    print(f'   {col:<22} {col_min:>8.4f} {col_max:>8.4f}{flag}')

# Save the scaler for inference reuse
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print(f'\n💾 Scaler saved → scaler.pkl')


✅ Scaling complete!
   Scaler fitted on 2,737 training rows only.

   Post-scaling ranges (training set):
   Feature                     Min      Max
   ────────────────────────────────────────
   volume                   0.0000   1.0000
   return                   0.0000   1.0000
   volume_change            0.0000   1.0000
   rsi14                    0.0000   1.0000
   bb_width                 0.0000   1.0000
   macd                     0.0000   1.0000
   signal                   0.0000   1.0000
   ma_cross_strength        0.0000   1.0000
   bb_position              0.0000   1.0000
   macd_hist                0.0000   1.0000
   ma5_dist                 0.0000   1.0000
   ma20_dist                0.0000   1.0000
   atr14                    0.0000   1.0000
   obv_change               0.0000   1.0000
   body_ratio               0.0000   1.0000
   day_of_week              0.0000   1.0000

💾 Scaler saved → scaler.pkl


<br><br>

### Part E: Sequence Window Creation & Save All Artifacts.


In [23]:
# ============================================================
#  E1 — Sliding Window Creation  (Parallel)
#
#  Converts the 2-D scaled matrix  (rows × features)
#  into 3-D windowed sequences     (samples × timesteps × features)
#  ready for the CNN + LSTM input layer.
#
#  Window logic:
#    For window size W, sample k is:
#      X[k] = feature_matrix[k : k+W]    shape: (W, n_features)
#      y[k] = label[k + W − 1]           target of last day in window
#
#  Why label at k+W−1 (not k+W)?
#    target[t] = 1 if adj_close[t+1] > adj_close[t]
#    So target[k+W-1] = 'will day k+W go up?'
#    Given 30 past days, predict direction of the NEXT day.
#
#  WINDOW_SIZE = 30  (changed from 20)
#    Why 30?
#    MACD = EMA12 − EMA26. EMA26 needs ~26 days to stabilise.
#    With a 20-day window the first 6 timesteps of each sample
#    contain partially-computed MACD values — the CNN is seeing
#    an artefact of the indicator's warmup, not the signal itself.
#    30 days gives MACD a full computation window inside every
#    sample. Also captures one full trading month + buffer,
#    and aligns better with the 20-day Bollinger Band period.
#    Cost: ~150 fewer training samples (3,780 → 3,630) — acceptable.
# ============================================================

WINDOW_SIZE = 30   # changed from 20 — gives MACD full computation window

def _make_windows_chunk(X_arr, y_arr, start, end, W):
    """Build windows for index range [start, end). Pure function — safe for multiprocessing."""
    X_chunk, y_chunk = [], []
    for i in range(start, end):
        X_chunk.append(X_arr[i : i + W])
        y_chunk.append(y_arr[i + W - 1])
    return (
        np.array(X_chunk, dtype=np.float32),
        np.array(y_chunk, dtype=np.int8)
    )


def make_windows_parallel(X_arr, y_arr, W, n_jobs):
    """Dispatch sliding window creation across n_jobs parallel workers."""
    n_samples  = len(X_arr) - W + 1
    chunk_size = max(1, n_samples // n_jobs)

    ranges = []
    for k in range(n_jobs):
        s = k * chunk_size
        e = min(s + chunk_size, n_samples)
        if s < n_samples:
            ranges.append((s, e))

    results = Parallel(n_jobs=n_jobs)(
        delayed(_make_windows_chunk)(X_arr, y_arr, s, e, W)
        for (s, e) in ranges
    )

    X_out = np.concatenate([r[0] for r in results], axis=0)
    y_out = np.concatenate([r[1] for r in results], axis=0)
    return X_out, y_out


print(f'⏳ Creating sliding windows  (W={WINDOW_SIZE})  using {N_CORES} parallel cores...')
print(f'   (Runs three passes: train → val → test)\n')

X_train, y_train = make_windows_parallel(X_train_scaled, y_train_raw, WINDOW_SIZE, N_CORES)
print(f'   ✅ Train windows done  →  X_train: {X_train.shape}')

X_val,   y_val   = make_windows_parallel(X_val_scaled,   y_val_raw,   WINDOW_SIZE, N_CORES)
print(f'   ✅ Val   windows done  →  X_val  : {X_val.shape}')

X_test,  y_test  = make_windows_parallel(X_test_scaled,  y_test_raw,  WINDOW_SIZE, N_CORES)
print(f'   ✅ Test  windows done  →  X_test : {X_test.shape}')

print(f'\n   Each sample shape : ({WINDOW_SIZE} timesteps  ×  {X_train.shape[2]} features)')
print(f'   Memory footprint  : X_train = {X_train.nbytes / 1e6:.2f} MB,'
      f'  X_val = {X_val.nbytes/1e6:.2f} MB,'
      f'  X_test = {X_test.nbytes/1e6:.2f} MB')

⏳ Creating sliding windows  (W=30)  using 15 parallel cores...
   (Runs three passes: train → val → test)

   ✅ Train windows done  →  X_train: (2700, 30, 16)
   ✅ Val   windows done  →  X_val  : (465, 30, 16)
   ✅ Test  windows done  →  X_test : (510, 30, 16)

   Each sample shape : (30 timesteps  ×  16 features)
   Memory footprint  : X_train = 5.18 MB,  X_val = 0.89 MB,  X_test = 0.98 MB


In [24]:
# ============================================================
#  E2 — Final Sanity Checks & Save All Artifacts
#
#  Before writing to disk, hard assertions verify:
#    1. All arrays are exactly 3-D with the right dimensions
#    2. No NaN or Inf values leaked into the windowed arrays
#    3. Class balance is preserved in each split's labels
#
#  Artifacts saved:
#    X_train.npy / y_train.npy  — training windows & labels
#    X_val.npy   / y_val.npy    — validation windows & labels
#    X_test.npy  / y_test.npy   — test windows & labels
#    scaler.pkl                  — fitted MinMaxScaler (from Cell D1)
#    feature_cols.pkl            — ordered list of feature names
#    preprocessing_meta.pkl      — all config values for traceability
#
#  The meta dict is a full audit trail so any future notebook
#  can load it and know exactly how the data was built.
# ============================================================

print('--- Sanity Checks ---')
if not os.path.exists('data'):
    os.makedirs('data')

# Shape assertions
for name, X, y in [('data/train', X_train, y_train),
                    ('data/val',   X_val,   y_val),
                    ('data/test',  X_test,  y_test)]:
    assert X.ndim   == 3,            f'X_{name} must be 3-D'
    assert X.shape[1] == WINDOW_SIZE, f'X_{name} timestep mismatch'
    assert X.shape[2] == len(FEATURE_COLS), f'X_{name} feature count mismatch'
    assert len(X) == len(y),        f'X_{name} / y_{name} length mismatch'
print('   ✅ Shape assertions passed')

# NaN / Inf check
for name, X in [('data/train', X_train), ('data/val', X_val), ('data/test', X_test)]:
    assert not np.isnan(X).any(), f'NaN found in X_{name}'
    assert not np.isinf(X).any(), f'Inf found in X_{name}'
print('   ✅ No NaN / Inf in any windowed array')

# Label balance per split
for name, y in [('data/train', y_train), ('data/val', y_val), ('data/test', y_test)]:
    up = y.mean()
    print(f'   ✅ y_{name:<5} balance — UP: {up:.2%}  DOWN: {1-up:.2%}')

# ── Save ──────────────────────────────────────────────────────
print('\n⏳ Saving all artifacts...')

np.save('data/X_train.npy', X_train)
np.save('data/y_train.npy', y_train)
np.save('data/X_val.npy',   X_val)
np.save('data/y_val.npy',   y_val)
np.save('data/X_test.npy',  X_test)
np.save('data/y_test.npy',  y_test)

with open('data/feature_cols.pkl', 'wb') as f:
    pickle.dump(FEATURE_COLS, f)

meta = {
    'window_size'    : WINDOW_SIZE,
    'n_features'     : len(FEATURE_COLS),
    'feature_cols'   : FEATURE_COLS,
    'train_date_range': (str(train_df.index.min().date()), str(train_df.index.max().date())),
    'val_date_range'  : (str(val_df.index.min().date()),   str(val_df.index.max().date())),
    'test_date_range' : (str(test_df.index.min().date()),  str(test_df.index.max().date())),
    'X_train_shape'  : tuple(X_train.shape),
    'X_val_shape'    : tuple(X_val.shape),
    'X_test_shape'   : tuple(X_test.shape),
    'scaler'         : 'MinMaxScaler(0,1) — fitted on training set only',
    'leakage_cols_dropped': ['open', 'high', 'low', 'close', 'adj_close'],
    'features_added' : ['ma_cross_strength', 'bb_position', 'macd_hist',
                        'atr14', 'obv', 'body_ratio', 'day_of_week'],
    'features_dropped': ['ma_cross', 'bb_upper', 'bb_lower', 'bb_mid'],
}
with open('data/preprocessing_meta.pkl', 'wb') as f:
    pickle.dump(meta, f)

SAVED = [
    'data/X_train.npy', 'data/y_train.npy',
    'data/X_val.npy',   'data/y_val.npy',
    'data/X_test.npy',  'data/y_test.npy',
    'data/scaler.pkl',  'data/feature_cols.pkl', 'data/preprocessing_meta.pkl'
]
print('\n✅ All artifacts saved:')
for fname in SAVED:
    print(f'   💾  {fname}')

print(f'\n{"="*54}')
print(f'  Chapter 2A complete — ready for model training.')
print(f'  CNN+LSTM input shape per sample: ({WINDOW_SIZE}, {len(FEATURE_COLS)})')
print(f'  Training samples : {len(X_train):,}')
print(f'  Val     samples  : {len(X_val):,}')
print(f'  Test    samples  : {len(X_test):,}')
print(f'{"="*54}')


--- Sanity Checks ---
   ✅ Shape assertions passed
   ✅ No NaN / Inf in any windowed array
   ✅ y_data/train balance — UP: 52.96%  DOWN: 47.04%
   ✅ y_data/val balance — UP: 50.32%  DOWN: 49.68%
   ✅ y_data/test balance — UP: 54.90%  DOWN: 45.10%

⏳ Saving all artifacts...

✅ All artifacts saved:
   💾  data/X_train.npy
   💾  data/y_train.npy
   💾  data/X_val.npy
   💾  data/y_val.npy
   💾  data/X_test.npy
   💾  data/y_test.npy
   💾  data/scaler.pkl
   💾  data/feature_cols.pkl
   💾  data/preprocessing_meta.pkl

  Chapter 2A complete — ready for model training.
  CNN+LSTM input shape per sample: (30, 16)
  Training samples : 2,700
  Val     samples  : 465
  Test    samples  : 510
